In [ ]:
from scipy.optimize import minimize
from scipy.optimize import NonlinearConstraint

In [ ]:
def rosenbrock(x):
    a = 100
    b = 1
    return a * (x[1] - x[0]**2)**2 + (b - x[0])**2

def constraint_eq(x):
    return x[0]**2 + x[1]**2

c = 1
constraint = NonlinearConstraint(constraint_eq, c, c)

x0 = [1,1]
result = minimize(rosenbrock, x0, method='trust-constr', constraints=constraint, jac='2-point')

print("Min point found: ", result.x)
print("Grad found: ", result.grad)

In [ ]:
import mujoco
from mujoco import mjx
import jax.numpy as jnp
from pathlib import Path

ur5epath = str(Path(Path.cwd()/'ur5e.xml'))
print (ur5epath)

mj_model = mujoco.MjModel.from_xml_path(ur5epath)
mj_data = mujoco.MjData(mj_model)
renderer = mujoco.Renderer(mj_model)
mjx_model = mjx.put_model(mj_model)
mjx_data = mjx.put_data(mj_model, mj_data)



In [ ]:
def forward_kinematics(q):
    new_mjx_data= mjx_data.replace(qpos=q)
    new_mjx_data= mjx.fwd_position(mjx_model, new_mjx_data)
    pos = new_mjx_data.site_xpos[mj_model.site('attachment_site').id]
    mat = new_mjx_data.site_xmat[mj_model.site('attachment_site').id]
    return pos, mat
print(forward_kinematics(jnp.zeros(6))[0])
print(forward_kinematics(jnp.zeros(6))[1])

In [ ]:
import scipy.optimize as optim
import jax

def pose_err(q, target_pos, target_mat):
    pos, mat = forward_kinematics(q)

    w_pos = 0.75
    w_rot = 0.25

    pos_err = jnp.sum((target_pos - pos) ** 2)
    rot_err = jnp.sum((target_mat - mat) ** 2)

    diff_scalar = w_pos * pos_err + w_rot * rot_err

    return diff_scalar


#jit compilation to make the run much faster
jit_err = jax.jit(pose_err)
#just get the gradient with auto-grad feature, with respect to the joint position guess (t
jit_err_grad = jax.jit(jax.grad(pose_err, argnums=(0)))

def ik_optim(target_pos, target_mat, init_guess):
    result = optim.minimize(jit_err, args=(target_pos, target_mat), x0=init_guess, method='trust-constr', jac=jit_err_grad)
    return result.x



In [ ]:
test_q = jnp.deg2rad(jnp.array([-90, -60, 90, -30, 0, 0]))
print(test_q.shape)
pos, rot = forward_kinematics(test_q)
print(pos, rot)
target_pos = pos + jnp.array([0, 0, 0.2])
target_rot = rot
target_q = ik_optim(target_pos, target_rot, init_guess=test_q)
print(target_q)
res_pos, res_rot = forward_kinematics(target_q)
print(res_pos)
print(res_rot)